## Fraud Detection In Bitcoin Transaction Network — 3rd Iteration

### What changed and why (vs 2nd Iteration)

| # | Change | Reason |
|---|--------|--------|
| 1 | **GATv2Conv** replaces GATConv | Dynamic attention (key-query depend on both nodes); stronger expressivity |
| 2 | **Residual connections** between layers | Mitigates over-smoothing; better gradient flow |
| 3 | **Self-loops + reverse edges** | Undirected representation; every node aggregates its own features |
| 4 | **Time-step sin/cos encoding + degree features** | Richer node context; temporal pattern without ordinal bias |
| 5 | **Log1p transform on skewed features** | Reduces dynamic range; stabilises training |
| 6 | **Cosine Annealing with Warm Restarts** | Escapes local minima; empirically beats ReduceLROnPlateau |
| 7 | **Combined Focal + pos_weight BCE** | Double-sided imbalance correction |
| 8 | **Ablation study** (GAT vs GAT+TDA) | Quantify TDA contribution |
| 9 | **PR-AUC-optimal threshold search** | Maximises F1 on val without test peeking |
| 10 | **4-layer GATv2 with wider head** | Extra capacity for complex patterns |


In [ ]:
# ================================
# INSTALL DEPENDENCIES
# ================================
!pip install -q kagglehub torch torchvision torchaudio
!pip install -q torch-geometric
!pip install -q pandas numpy scikit-learn networkx matplotlib seaborn
!pip install -q ripser persim


In [ ]:
# ================================
# GLOBAL SEEDS — REPRODUCIBILITY
# ================================
import random, os
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", DEVICE)


In [ ]:
# ================================
# IMPORTS
# ================================
import pandas as pd
import torch.nn.functional as F
from torch import nn

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer

import networkx as nx
from torch_geometric.nn import GATv2Conv
from torch_geometric.utils import add_self_loops, to_undirected

import matplotlib.pyplot as plt
import seaborn as sns
from ripser import ripser

print("Imports OK")


In [ ]:
# ================================
# LOAD DATA
# ================================
import kagglehub

path = kagglehub.dataset_download("ellipticco/elliptic-data-set")
base_path = os.path.join(path, "elliptic_bitcoin_dataset")

features = pd.read_csv(f"{base_path}/elliptic_txs_features.csv", header=None)
edges    = pd.read_csv(f"{base_path}/elliptic_txs_edgelist.csv")
classes  = pd.read_csv(f"{base_path}/elliptic_txs_classes.csv")

print("Features:", features.shape)
print("Edges:   ", edges.shape)
print("Classes: ", classes.shape)


In [ ]:
# ================================
# PREPROCESS
# ================================
features.columns = ["txId", "time_step"] + [f"f{i}" for i in range(165)]

data = features.merge(classes, on="txId")

# Drop unlabelled nodes
data = data[data["class"] != "unknown"].copy()
data["class"] = data["class"].map({"1": 1, "2": 0})

time_step_vals = data["time_step"].values
X_raw = data.drop(["txId", "time_step", "class"], axis=1).values.astype(float)
y     = data["class"].values.astype(int)
tx_ids = data["txId"].values

print(f"Labelled nodes: {len(y)}  |  Illicit: {y.sum()}  |  Licit: {(1-y).sum()}")
print(f"Imbalance ratio (licit/illicit): {(1-y).sum()/y.sum():.1f}")


In [ ]:
# ================================
# FEATURE ENGINEERING
# Improvements vs 2nd iteration:
#  1. time_step sin/cos encoding (avoids ordinal bias)
#  2. Log1p transform for skewed raw features
#  3. Node degree features from the graph
# ================================

# ── 1. Time-step sin/cos encoding ─────────────────────────────────────
T = time_step_vals.max()
ts_sin = np.sin(2 * np.pi * time_step_vals / T).reshape(-1, 1)
ts_cos = np.cos(2 * np.pi * time_step_vals / T).reshape(-1, 1)
ts_norm = (time_step_vals / T).reshape(-1, 1)   # linear normalised (kept too)
time_features = np.concatenate([ts_norm, ts_sin, ts_cos], axis=1)  # (N, 3)

# ── 2. Log1p transform on raw features (reduce skew) ─────────────────
X_log = np.sign(X_raw) * np.log1p(np.abs(X_raw))

# ── 3. Graph construction (needed for degree features) ───────────────
valid_nodes = set(data["txId"])
edge_list = [
    (r["txId1"], r["txId2"])
    for _, r in edges.iterrows()
    if r["txId1"] in valid_nodes and r["txId2"] in valid_nodes
]
G = nx.DiGraph()
G.add_edges_from(edge_list)
node_map = {tx: i for i, tx in enumerate(tx_ids)}

in_deg  = np.array([G.in_degree(tx)  if G.has_node(tx) else 0 for tx in tx_ids], dtype=float)
out_deg = np.array([G.out_degree(tx) if G.has_node(tx) else 0 for tx in tx_ids], dtype=float)
tot_deg = in_deg + out_deg
deg_features = np.stack([in_deg, out_deg, tot_deg,
                          np.log1p(in_deg), np.log1p(out_deg)], axis=1)  # (N, 5)

# ── 4. Stack all raw features ────────────────────────────────────────
X_combined = np.concatenate([X_raw, X_log, time_features, deg_features], axis=1)
print(f"Feature matrix shape before TDA: {X_combined.shape}")


In [ ]:
# ================================
# GRAPH: edge_index
# • Convert to undirected (add reverse edges)
# • Add self-loops
# ================================
edge_index_list = [
    [node_map[u], node_map[v]]
    for u, v in edge_list
]
edge_index = torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()

n_nodes = len(node_map)

# Make undirected (adds reverse edges, deduplicates)
edge_index = to_undirected(edge_index, num_nodes=n_nodes)

# Add self-loops so each node can aggregate its own features
edge_index, _ = add_self_loops(edge_index, num_nodes=n_nodes)

print(f"Nodes: {n_nodes}  |  Edges (undirected + self-loops): {edge_index.shape[1]}")


In [ ]:
# ================================
# TDA FEATURES (all labelled nodes)
# ================================

def compute_tda(subgraph):
    nodes = list(subgraph.nodes())
    if len(nodes) < 3:
        return np.zeros(6)
    try:
        A    = nx.to_numpy_array(subgraph)
        dist = 1.0 - A
        np.fill_diagonal(dist, 0)
        dgms = ripser(dist, distance_matrix=True, maxdim=1)['dgms']
        h0   = dgms[0]
        h1   = dgms[1]

        h0_pers = h0[:, 1] - h0[:, 0]
        h0_pers = h0_pers[np.isfinite(h0_pers)]

        if len(h1) == 0:
            return np.array([
                len(h0_pers),
                h0_pers.max()  if len(h0_pers) else 0,
                h0_pers.mean() if len(h0_pers) else 0,
                0, 0, 0
            ])

        h1_pers = h1[:, 1] - h1[:, 0]
        return np.array([
            len(h0_pers),
            h0_pers.max()  if len(h0_pers) else 0,
            h0_pers.mean() if len(h0_pers) else 0,
            len(h1_pers),
            h1_pers.max()  if len(h1_pers) else 0,
            h1_pers.mean() if len(h1_pers) else 0,
        ])
    except Exception:
        return np.zeros(6)


print("Computing TDA features for ALL labelled nodes …")
tda_full = np.zeros((len(tx_ids), 6))
for tx in tx_ids:
    try:
        sub_nodes = list(
            nx.single_source_shortest_path_length(G, tx, cutoff=2).keys()
        )
        subgraph = G.subgraph(sub_nodes)
        tda_full[node_map[tx]] = compute_tda(subgraph)
    except Exception:
        pass

# Combine all features
X = np.concatenate([X_combined, tda_full], axis=1)
# Also keep a copy without TDA for ablation
X_no_tda = X_combined.copy()

print(f"Feature matrix shape (with TDA):    {X.shape}")
print(f"Feature matrix shape (without TDA): {X_no_tda.shape}")


In [ ]:
# ================================
# CLEAN FEATURES
# ================================
def clean_features(arr):
    arr = arr.copy()
    arr[np.isinf(arr)] = np.nan
    arr = np.clip(arr, -1e6, 1e6)
    arr = np.nan_to_num(arr, nan=0.0)
    return arr

X       = clean_features(X)
X_no_tda = clean_features(X_no_tda)

print("NaN count:", np.isnan(X).sum(), "| Inf count:", np.isinf(X).sum())


In [ ]:
# ================================
# TRAIN / VAL / TEST SPLIT
# Stratified 60 / 20 / 20
# Scaler fitted on TRAIN ONLY → no leakage
# ================================
idx_all = np.arange(len(y))

idx_tv, test_idx = train_test_split(
    idx_all, test_size=0.20, random_state=SEED, stratify=y
)
train_idx, val_idx = train_test_split(
    idx_tv, test_size=0.25, random_state=SEED, stratify=y[idx_tv]
)

print(f"Train: {len(train_idx)}  Val: {len(val_idx)}  Test: {len(test_idx)}")

def scale_split(X_arr):
    X_s = X_arr.copy()
    sc  = StandardScaler()
    X_s[train_idx] = sc.fit_transform(X_s[train_idx])
    X_s[val_idx]   = sc.transform(X_s[val_idx])
    X_s[test_idx]  = sc.transform(X_s[test_idx])
    return X_s

X        = scale_split(X)
X_no_tda = scale_split(X_no_tda)

X_tensor       = torch.tensor(X,       dtype=torch.float).to(DEVICE)
X_notda_tensor = torch.tensor(X_no_tda, dtype=torch.float).to(DEVICE)
y_tensor       = torch.tensor(y,       dtype=torch.float).to(DEVICE)
edge_index     = edge_index.to(DEVICE)

print("Tensors ready on", DEVICE)


In [ ]:
# ================================
# FOCAL LOSS (with pos_weight)
# Combines focal modulation with class-weight correction.
# alpha  : scalar weight for positive class
# gamma  : focusing parameter (0 = plain BCE)
# pos_wt : extra multiplicative weight for positives (like BCEWithLogitsLoss)
# ================================

class FocalLoss(nn.Module):
    def __init__(self, alpha: float = 0.75, gamma: float = 2.0, pos_wt: float = None):
        super().__init__()
        self.alpha   = alpha
        self.gamma   = gamma
        self.pos_wt  = pos_wt   # optional manual pos_weight

    def forward(self, logits, targets):
        if self.pos_wt is not None:
            # Per-sample weights: positives get weight=pos_wt, negatives weight=1.0
            # Scales each class loss contribution, complementing focal modulation.
            weight = targets * self.pos_wt + (1 - targets)
            bce = F.binary_cross_entropy_with_logits(
                logits, targets, weight=weight, reduction='none')
        else:
            bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p_t     = torch.sigmoid(logits) * targets + (1 - torch.sigmoid(logits)) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        loss    = alpha_t * (1 - p_t) ** self.gamma * bce
        return loss.mean()


# Auto-compute pos_weight from training labels
n_neg = (y[train_idx] == 0).sum()
n_pos = (y[train_idx] == 1).sum()
AUTO_POS_WT = n_neg / n_pos
print(f"pos_weight = {AUTO_POS_WT:.2f}  (licit/illicit ratio in train)")
print("FocalLoss defined")


In [ ]:
# ================================
# IMPROVED GATv2 MODEL
# Changes vs 2nd iteration:
#   • GATv2Conv (dynamic attention)
#   • Residual connections (layers 2→3, 3→4)
#   • 4 GAT layers
#   • BatchNorm + ELU throughout
#   • MLP classifier head with skip connection
# ================================

class GATv2Net(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, heads=8, dropout=0.3):
        super().__init__()
        self.drop = dropout
        h = hidden_dim * heads   # concatenated head output dim

        # Input projection for residual shortcut (dim matching)
        self.proj1 = nn.Linear(in_dim, h, bias=False)

        # Layer 1: in_dim → h
        self.gat1 = GATv2Conv(in_dim, hidden_dim, heads=heads,
                               dropout=dropout, concat=True)
        self.bn1  = nn.BatchNorm1d(h)

        # Layer 2: h → h  (residual: out + in)
        self.gat2 = GATv2Conv(h, hidden_dim, heads=heads,
                               dropout=dropout, concat=True)
        self.bn2  = nn.BatchNorm1d(h)

        # Layer 3: h → h  (residual: out + in)
        self.gat3 = GATv2Conv(h, hidden_dim, heads=heads,
                               dropout=dropout, concat=True)
        self.bn3  = nn.BatchNorm1d(h)

        # Layer 4: h → hidden_dim  (average heads, no concat)
        self.gat4 = GATv2Conv(h, hidden_dim, heads=1,
                               dropout=dropout, concat=False)
        self.bn4  = nn.BatchNorm1d(hidden_dim)

        # MLP classifier head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 32),
            nn.ELU(),
            nn.Linear(32, 1)
        )

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.drop, training=self.training)

        # Layer 1
        x1 = self.gat1(x, edge_index)
        x1 = self.bn1(x1)
        x1 = F.elu(x1) + self.proj1(x)   # pre-activation residual shortcut
        x1 = F.dropout(x1, p=self.drop, training=self.training)

        # Layer 2 (residual)
        x2 = self.gat2(x1, edge_index)
        x2 = self.bn2(x2)
        x2 = F.elu(x2) + x1   # pre-activation residual
        x2 = F.dropout(x2, p=self.drop, training=self.training)

        # Layer 3 (residual)
        x3 = self.gat3(x2, edge_index)
        x3 = self.bn3(x3)
        x3 = F.elu(x3) + x2   # pre-activation residual
        x3 = F.dropout(x3, p=self.drop, training=self.training)

        # Layer 4
        x4 = self.gat4(x3, edge_index)
        x4 = self.bn4(x4)
        x4 = F.elu(x4)
        x4 = F.dropout(x4, p=self.drop, training=self.training)

        return self.classifier(x4).squeeze(-1)   # raw logits


print("GATv2Net defined")


In [ ]:
# ================================
# TRAINING LOOP
# Changes vs 2nd iteration:
#   • CosineAnnealingWarmRestarts (escapes local minima)
#   • PR-AUC as early-stopping metric (better for imbalance)
#   • pos_weight auto-derived from training set
#   • Configurable feature tensor (for ablation)
# ================================

def train_gatv2(in_dim, hidden_dim=128, heads=8, dropout=0.3,
                lr=3e-3, weight_decay=1e-4,
                focal_alpha=0.75, focal_gamma=2.0,
                max_epochs=600, patience=60,
                T0=50, T_mult=2,
                feat_tensor=None, verbose=True):

    if feat_tensor is None:
        feat_tensor = X_tensor

    model     = GATv2Net(in_dim, hidden_dim, heads, dropout).to(DEVICE)
    criterion = FocalLoss(alpha=focal_alpha, gamma=focal_gamma,
                          pos_wt=AUTO_POS_WT)
    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=lr, weight_decay=weight_decay)
    # Cosine Annealing with Warm Restarts
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=T0, T_mult=T_mult, eta_min=1e-6
    )

    best_val_metric = -1.0
    patience_count  = 0
    best_state      = None
    history         = []

    for epoch in range(max_epochs):
        # ── Train ──────────────────────────────────────────────
        model.train()
        optimizer.zero_grad()
        logits = model(feat_tensor, edge_index)
        loss   = criterion(logits[train_idx], y_tensor[train_idx])

        if torch.isnan(loss) or torch.isinf(loss):
            print(f"Invalid loss at epoch {epoch}. Stopping.")
            break

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        scheduler.step()   # called once per epoch; advances by 1 epoch internally

        # ── Validate ───────────────────────────────────────────
        model.eval()
        with torch.no_grad():
            val_logits = model(feat_tensor, edge_index)
            val_probs  = torch.sigmoid(val_logits).cpu().numpy()

        # Use PR-AUC as early-stopping signal (better for imbalance)
        val_prauc = average_precision_score(y[val_idx], val_probs[val_idx])

        history.append({'epoch': epoch, 'loss': loss.item(), 'val_prauc': val_prauc})

        if val_prauc > best_val_metric:
            best_val_metric = val_prauc
            patience_count  = 0
            best_state      = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_count += 1

        if patience_count >= patience:
            if verbose:
                print(f"Early stopping @ epoch {epoch}  best val PR-AUC={best_val_metric:.4f}")
            break

        if verbose and epoch % 25 == 0:
            print(f"Epoch {epoch:4d}  loss={loss.item():.4f}  "
                  f"val_PR-AUC={val_prauc:.4f}  "
                  f"lr={optimizer.param_groups[0]['lr']:.2e}")

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, best_val_metric, history


print("train_gatv2 defined")


In [ ]:
# ================================
# TRAIN MAIN MODEL (with TDA)
# ================================
print("Training GATv2 (with TDA features) …")
model, best_prauc, history = train_gatv2(X_tensor.shape[1])
print(f"\nBest validation PR-AUC: {best_prauc:.4f}")

# ── Plot training history ──────────────────────────────────────────────
epochs_hist = [h['epoch'] for h in history]
prauc_hist  = [h['val_prauc'] for h in history]

plt.figure(figsize=(8, 3))
plt.plot(epochs_hist, prauc_hist)
plt.xlabel("Epoch")
plt.ylabel("Val PR-AUC")
plt.title("Training History — GATv2 (TDA)")
plt.tight_layout()
plt.show()


In [ ]:
# ================================
# THRESHOLD TUNING — VALIDATION SET ONLY
# Search for threshold that maximises val F1
# ================================
model.eval()
with torch.no_grad():
    all_logits = model(X_tensor, edge_index)
    all_probs  = torch.sigmoid(all_logits).cpu().numpy()

thresholds  = np.arange(0.02, 0.98, 0.01)
best_thresh = 0.5
best_f1_val = 0.0

for t in thresholds:
    pred_v = (all_probs[val_idx] > t).astype(int)
    f1_v   = f1_score(y[val_idx], pred_v, zero_division=0)
    if f1_v > best_f1_val:
        best_f1_val = f1_v
        best_thresh = t

print(f"Best threshold (val):  {best_thresh:.2f}")
print(f"Val F1 at best thresh: {best_f1_val:.4f}")


In [ ]:
# ================================
# FINAL EVALUATION — TEST SET
# ================================
pred_test = (all_probs[test_idx] > best_thresh).astype(int)

print("=" * 60)
print("  IMPROVED GATv2 + TDA  —  TEST SET RESULTS")
print("=" * 60)
print(f"  Threshold : {best_thresh:.2f}")
print(f"  Accuracy  : {accuracy_score(y[test_idx], pred_test):.4f}")
print(f"  Precision : {precision_score(y[test_idx], pred_test, zero_division=0):.4f}")
print(f"  Recall    : {recall_score(y[test_idx], pred_test, zero_division=0):.4f}")
print(f"  F1        : {f1_score(y[test_idx], pred_test, zero_division=0):.4f}")
print(f"  ROC-AUC   : {roc_auc_score(y[test_idx], all_probs[test_idx]):.4f}")
print(f"  PR-AUC    : {average_precision_score(y[test_idx], all_probs[test_idx]):.4f}")
print("=" * 60)
print()
print(classification_report(
    y[test_idx], pred_test,
    target_names=["Licit (0)", "Illicit (1)"],
    zero_division=0
))

# ── Confusion matrix ───────────────────────────────────────────────────
cm = confusion_matrix(y[test_idx], pred_test)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Licit', 'Illicit'],
            yticklabels=['Licit', 'Illicit'])
plt.title('Confusion Matrix — GATv2 + TDA (3rd Iter.)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()


In [ ]:
# ================================
# ROC & PR CURVES
# ================================
fpr, tpr, _  = roc_curve(y[test_idx], all_probs[test_idx])
prec, rec, _ = precision_recall_curve(y[test_idx], all_probs[test_idx])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(fpr, tpr, label=f"ROC-AUC = {roc_auc_score(y[test_idx], all_probs[test_idx]):.4f}")
axes[0].plot([0, 1], [0, 1], 'k--')
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve — GATv2 + TDA")
axes[0].legend()

axes[1].plot(rec, prec, label=f"PR-AUC = {average_precision_score(y[test_idx], all_probs[test_idx]):.4f}")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve — GATv2 + TDA")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# ================================
# ABLATION STUDY
# Compare: GATv2+TDA  vs  GATv2 (no TDA)
# Both use the same architecture and seed.
# ================================
print("Training GATv2 WITHOUT TDA features …")
model_no_tda, prauc_no_tda, _ = train_gatv2(
    X_notda_tensor.shape[1],
    feat_tensor=X_notda_tensor,
    verbose=False
)

model_no_tda.eval()
with torch.no_grad():
    logits_nt = model_no_tda(X_notda_tensor, edge_index)
    probs_nt  = torch.sigmoid(logits_nt).cpu().numpy()

# Threshold tuning on val set
best_t_nt, best_f1_nt = 0.5, 0.0
for t in np.arange(0.02, 0.98, 0.01):
    f = f1_score(y[val_idx], (probs_nt[val_idx] > t).astype(int), zero_division=0)
    if f > best_f1_nt:
        best_f1_nt, best_t_nt = f, t

pred_nt = (probs_nt[test_idx] > best_t_nt).astype(int)

print("\n" + "=" * 60)
print("ABLATION STUDY — TEST SET")
print("=" * 60)
print(f"{'Model':<25} {'Acc':>7} {'F1':>7} {'ROC-AUC':>9} {'PR-AUC':>8}")
print("-" * 60)

# GATv2 + TDA
acc_tda  = accuracy_score(y[test_idx], pred_test)
f1_tda   = f1_score(y[test_idx], pred_test, zero_division=0)
roc_tda  = roc_auc_score(y[test_idx], all_probs[test_idx])
pr_tda   = average_precision_score(y[test_idx], all_probs[test_idx])
print(f"{'GATv2 + TDA':<25} {acc_tda:>7.4f} {f1_tda:>7.4f} {roc_tda:>9.4f} {pr_tda:>8.4f}")

# GATv2 without TDA
acc_nt  = accuracy_score(y[test_idx], pred_nt)
f1_nt   = f1_score(y[test_idx], pred_nt, zero_division=0)
roc_nt  = roc_auc_score(y[test_idx], probs_nt[test_idx])
pr_nt   = average_precision_score(y[test_idx], probs_nt[test_idx])
print(f"{'GATv2 (no TDA)':<25} {acc_nt:>7.4f} {f1_nt:>7.4f} {roc_nt:>9.4f} {pr_nt:>8.4f}")

print("-" * 60)
print(f"TDA lift (F1)   : {f1_tda - f1_nt:+.4f}")
print(f"TDA lift (PRAUC): {pr_tda - pr_nt:+.4f}")


In [ ]:
# ================================
# LIGHTWEIGHT HYPERPARAMETER SEARCH
# ================================
hp_grid = [
    dict(hidden_dim=128, heads=8,  dropout=0.3, lr=3e-3),
    dict(hidden_dim=128, heads=8,  dropout=0.4, lr=5e-3),
    dict(hidden_dim=256, heads=4,  dropout=0.3, lr=3e-3),
    dict(hidden_dim=128, heads=4,  dropout=0.5, lr=1e-3),
    dict(hidden_dim=192, heads=6,  dropout=0.3, lr=3e-3),
]

results = []
for cfg in hp_grid:
    print(f"\nConfig: {cfg}")
    m, vp, _ = train_gatv2(
        X_tensor.shape[1],
        hidden_dim   = cfg['hidden_dim'],
        heads        = cfg['heads'],
        dropout      = cfg['dropout'],
        lr           = cfg['lr'],
        verbose      = False,
        max_epochs   = 400,
        patience     = 50,
    )
    m.eval()
    with torch.no_grad():
        p = torch.sigmoid(m(X_tensor, edge_index)).cpu().numpy()
    # Val threshold search
    best_t, best_f = 0.5, 0.0
    for t in np.arange(0.02, 0.98, 0.02):
        f = f1_score(y[val_idx], (p[val_idx] > t).astype(int), zero_division=0)
        if f > best_f:
            best_f, best_t = f, t
    val_prauc_hp = average_precision_score(y[val_idx], p[val_idx])
    results.append({**cfg, 'val_f1': best_f, 'val_prauc': val_prauc_hp, 'best_thresh': best_t})
    print(f"  → val F1={best_f:.4f}  val PR-AUC={val_prauc_hp:.4f}  thresh={best_t:.2f}")

print("\n" + "=" * 65)
print("HYPERPARAMETER SEARCH SUMMARY (sorted by val PR-AUC)")
print("=" * 65)
for r in sorted(results, key=lambda x: x['val_prauc'], reverse=True):
    print(r)


In [ ]:
# ================================
# BASELINE: Random Forest comparison
# (on the same scaled features)
# ================================
imp = SimpleImputer(strategy='mean')
X_train_np = imp.fit_transform(X[train_idx])
X_val_np   = imp.transform(X[val_idx])
X_test_np  = imp.transform(X[test_idx])

print("Training Random Forest …")
rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED,
                             class_weight='balanced', max_depth=20)
rf.fit(X_train_np, y[train_idx])
pred_rf   = rf.predict(X_test_np)
proba_rf  = rf.predict_proba(X_test_np)[:, 1]

print("\nRandom Forest — Test Set:")
print(f"  Accuracy : {accuracy_score(y[test_idx], pred_rf):.4f}")
print(f"  F1       : {f1_score(y[test_idx], pred_rf, zero_division=0):.4f}")
print(f"  ROC-AUC  : {roc_auc_score(y[test_idx], proba_rf):.4f}")
print(f"  PR-AUC   : {average_precision_score(y[test_idx], proba_rf):.4f}")


## Results Summary

| Model | Accuracy | F1 | ROC-AUC | PR-AUC |
|-------|----------|----|---------|--------|
| Baseline GAT + TDA (1st iter) | ~96–97% | ~0.85–0.87 | ~0.98 | — |
| Improved GAT + TDA (2nd iter) | *see above* | *see above* | *see above* | — |
| **GATv2 + TDA (this run)** | *see above* | *see above* | *see above* | *see above* |
| GATv2 (no TDA) — ablation | *see above* | *see above* | *see above* | *see above* |
| Random Forest (baseline) | *see above* | *see above* | *see above* | *see above* |

### Key improvements (3rd iteration)

1. **GATv2Conv** — dynamic attention where the attention weight depends on the concatenation of both endpoint features projected through a learned linear layer, making it strictly more expressive than the original GAT.
2. **Residual connections** — help gradient flow in deep networks and reduce over-smoothing.  Self-loop + undirected edges ensure every node can aggregate its own information.
3. **Log1p transform** — reduces feature skewness common in financial transaction data.
4. **Time-step sin/cos encoding** — captures cyclical temporal pattern without imposing ordinal assumptions.
5. **Degree features** — in-degree/out-degree are strong signals in fraud networks (illicit nodes often have unusual connectivity patterns).
6. **CosineAnnealingWarmRestarts** — the periodic LR restarts help escape sharp local minima, often outperforming ReduceLROnPlateau on graph tasks.
7. **PR-AUC early stopping** — more reliable than F1@0.5 for highly imbalanced datasets.
8. **Combined Focal + pos_weight** — doubly down-weights easy negatives while also balancing the gradient magnitudes.

### If >98.5% accuracy is not reached

The Elliptic dataset remains inherently difficult due to low illicit prevalence (~10%) and temporal drift.  Remaining levers:
- Temporal GNN (e.g. EvolveGCN) that explicitly models graph dynamics over time steps
- Larger pre-training on the unlabelled nodes with self-supervised objectives
- Ensemble of GATv2 + Random Forest (soft probability averaging)
- Edge-level features (transaction amount, fee rate) if available
